# `filter` — Comprehensive Reference

`filter` filters a DataFrame using intuitive CLI-style string expressions or programmatic dictionaries.

Under the hood, `filter` compiles conditions into high-performance Polars expressions evaluated directly in native C++/Rust memory.

### Calling Styles

`filter` supports two clean, unambiguous calling styles:

- **String expressions** (matches CLI `-filter` syntax; handles spaces directly):
  `df.filter("[bill length mm] > 40 and species == 'Adelie'")`
- **Dictionaries** (ideal for dynamic programmatic keys and Option 1 / Option 2 ranges):
  `df.filter({"bill length mm": "> 40", "species": "Adelie"})`
  `df.filter({"body_mass_g": ("between", 3500, 4000)})`
  `df.filter({"body_mass_g": ((">=", 3500), ("<=", 4000))})`
- **Top-level library verb**:
  `pt.filter(df, "species == 'Adelie' and body_mass_g > 5000")`

> [!IMPORTANT]
> **Why `pytae` Strictly Enforces `==` and Disallows Kwargs**:
> In Python syntax, keyword arguments like `df.filter(species="Adelie")` force the use of a single `=`. In `pytae`, single `=` is reserved exclusively for column assignments in `-mutate`. To prevent subtle bugs and ensure **"one and only one obvious way"**, `filter()` strictly requires `==` and raises an immediate `TypeError` if keyword arguments are passed.

---

### Comprehensive Reference Table

| Category | Example Syntax | Meaning / Target |
|---|---|---|
| **Equality** | `"species == 'Adelie'"` or `{"species": "Adelie"}` | Exact equality match (`==` strictly enforced; single `=` raises error) |
| **Inequality** | `"species != 'Adelie'"` or `{"species": ("!=", "Adelie")}` | Inequality match |
| **Relational Operators** | `"body_mass_g > 5000"` or `{"body_mass_g": (">", 5000)}` | `>`, `>=`, `<`, `<=`, `==`, `!=` |
| **Set Membership** | `"species in ['Adelie', 'Gentoo']"` or `{"species": ['Adelie', 'Gentoo']}` | In candidate set / list |
| **Exclusion** | `"sex not in ['Male']"` or `{"sex": ("not in", ['Male'])}` | Not in candidate set |
| **Boolean AND** | `"species == 'Gentoo' and body_mass_g > 4500"` | Logical conjunction (comma `,` and bitwise `&` disallowed) |
| **Boolean OR** | `"island == 'Dream' or island == 'Torgersen'"` | Logical disjunction (bitwise `\|` disallowed) |
| **Grouping / Precedence** | `"(species == 'Adelie' or species == 'Chinstrap') and mass > 4000"` | Arbitrary nested boolean logic with parentheses |
| **Range Option 1 (`between`)** | `{"body_mass_g": ("between", 3500, 4000)}` | Inclusive range (`min <= x <= max`) via Polars `.is_between()` |
| **Range Option 2 (operator tuples)** | `{"body_mass_g": ((">=", 3500), ("<=", 4000))}` | Multiple conditions on same column via clean `(op, val)` pairs |
| **String Range Expression** | `"body_mass_g >= 3500 and body_mass_g <= 4000"` | Explicit boolean bounds in string expression |
| **Columns with Spaces** | `"[bill length mm] > 40"` or `{"bill length mm": (">", 40)}` | Spaced column syntax (brackets in strings, plain keys in dicts) |
| **String Prefixes/Suffixes** | `{"species": ("starts_with", "Ad")}` or `{"species": ("ends_with", "strap")}` | String prefix / suffix search |
| **Multiple Prefixes/Suffixes** | `{"species": ("starts_with", ["Ad", "Ge"])}` | Matches any prefix / suffix in sequence |
| **String Substring / Regex** | `{"species": ("contains", "adel")}` or `{"species": ("regex", r"oo")}` | Substring match or regex search |
| **Null Checks** | `"sex.is_not_null()"`, `{"sex": "is_null()"}` | Universal method tokens (`is_null()`, `is_not_null()`); all missing and NaN values normalize to null |
| **Column-to-Column** | `"exposure > limit"` or `"tranche_id == deal_id"` | Compare two columns using bare column names |
| **Variable Injection** | `"exposure > @threshold and tranche == col(@target_col)"` | External scalar (`@var`) and column (`col(@var)`) references |

---

In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "src"))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt
from pytae import c

penguins = pt.sample_data["penguins"]
tips = pt.sample_data["tips"]
titanic = pt.sample_data["titanic"]
print("Loaded sample datasets:", list(pt.sample_data.keys()))


Loaded sample datasets: ['anagrams', 'anscombe', 'attention', 'brain_networks', 'car_crashes', 'diamonds', 'dots', 'dowjones', 'exercise', 'flights', 'fmri', 'geyser', 'glue', 'healthexp', 'iris', 'mpg', 'penguins', 'planets', 'seaice', 'taxis', 'tips', 'titanic']


## 1. Primary Calling Styles: String Expressions and Dictionaries

`filter()` accepts conditions as string expressions matching the CLI `-filter` syntax, or plain dictionaries for programmatic filters. Both direct method chaining (`df.filter(...)`) and the top-level library verb (`pt.filter(df, ...)`) are supported.

In [2]:
# 1. Canonical c expressions — clean, typed, composable
(
    penguins
    .filter(c.species == "Adelie", c.island == "Dream")
    .head(5)
)


species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
str,str,f64,f64,f64,f64,str
"""Adelie""","""Dream""",39.5,16.7,178.0,3250.0,"""Female"""
"""Adelie""","""Dream""",37.2,18.1,178.0,3900.0,"""Male"""
"""Adelie""","""Dream""",39.5,17.8,188.0,3300.0,"""Female"""
"""Adelie""","""Dream""",40.9,18.9,184.0,3900.0,"""Male"""
"""Adelie""","""Dream""",36.4,17.0,195.0,3325.0,"""Female"""


In [3]:
# 2. Composing multiple conditions with bitwise operators (&, |)
(
    penguins
    .filter((c.species == "Adelie") & (c.island == "Dream"))
    .head(5)
)


species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
str,str,f64,f64,f64,f64,str
"""Adelie""","""Dream""",39.5,16.7,178.0,3250.0,"""Female"""
"""Adelie""","""Dream""",37.2,18.1,178.0,3900.0,"""Male"""
"""Adelie""","""Dream""",39.5,17.8,188.0,3300.0,"""Female"""
"""Adelie""","""Dream""",40.9,18.9,184.0,3900.0,"""Male"""
"""Adelie""","""Dream""",36.4,17.0,195.0,3325.0,"""Female"""


In [4]:
# 3. Composing multiple conditions — either chained or passed as multiple positional arguments
(
    penguins
    .filter(c.body_mass_g > 4000, c.island == "Biscoe")
    .head(5)
)


species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
str,str,f64,f64,f64,f64,str
"""Adelie""","""Biscoe""",40.1,18.9,188.0,4300.0,"""Male"""
"""Adelie""","""Biscoe""",42.0,19.5,200.0,4050.0,"""Male"""
"""Adelie""","""Biscoe""",41.3,21.1,195.0,4400.0,"""Male"""
"""Adelie""","""Biscoe""",41.1,18.2,192.0,4050.0,"""Male"""
"""Adelie""","""Biscoe""",41.1,19.1,188.0,4100.0,"""Male"""


In [5]:
# 4. Top-level function pt.filter(df, ...) — works identically to DataFrame method
pt.filter(penguins, c.species == "Gentoo", c.body_mass_g > 5000).head(5)


species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
str,str,f64,f64,f64,f64,str
"""Gentoo""","""Biscoe""",50.0,16.3,230.0,5700.0,"""Male"""
"""Gentoo""","""Biscoe""",50.0,15.2,218.0,5700.0,"""Male"""
"""Gentoo""","""Biscoe""",47.6,14.5,215.0,5400.0,"""Male"""
"""Gentoo""","""Biscoe""",46.7,15.3,219.0,5200.0,"""Male"""
"""Gentoo""","""Biscoe""",46.8,15.4,215.0,5150.0,"""Male"""


## 2. Relational Comparisons (`==`, `!=`, `<`, `>`, `<=`, `>=`)

Comparison filters can be written directly in string expressions or as operator tuples inside dictionaries:
- String expressions: `"body_mass_g > 5000"`, `"species == 'Gentoo'"`
- Dictionary with operator tuples: `{"body_mass_g": (">", 5000)}`, `{"island": ("!=", "Dream")}`

> [!IMPORTANT]
> **Strict Equality Enforcement**: In `pytae`, equality comparisons in filter strings strictly require `==`. Single `=` is reserved exclusively for column assignments in `-mutate`. Using single `=` in filter expressions raises an immediate diagnostic error.

In [6]:
# Numeric comparisons via c expressions
(
    penguins
    .filter(c.body_mass_g > 5000, c.flipper_length_mm >= 220)
    .select(c.species, c.island, c.body_mass_g, c.flipper_length_mm)
    .head(5)
)


species,island,body_mass_g,flipper_length_mm
str,str,f64,f64
"""Gentoo""","""Biscoe""",5700.0,230.0
"""Gentoo""","""Biscoe""",6300.0,221.0
"""Gentoo""","""Biscoe""",5350.0,222.0
"""Gentoo""","""Biscoe""",5550.0,220.0
"""Gentoo""","""Biscoe""",5250.0,222.0


In [7]:
# Inequality and comparison conditions
(
    penguins
    .filter(c.body_mass_g > 5500, c.island != "Dream")
    .select(c.species, c.island, c.body_mass_g)
    .head(5)
)


species,island,body_mass_g
str,str,f64
"""Gentoo""","""Biscoe""",5700.0
"""Gentoo""","""Biscoe""",5700.0
"""Gentoo""","""Biscoe""",5550.0
"""Gentoo""","""Biscoe""",5850.0
"""Gentoo""","""Biscoe""",5850.0


In [8]:
# Multiple comparisons in c expressions
(
    penguins
    .filter(c.body_mass_g <= 3000, c.bill_length_mm < 37)
    .select(c.species, c.island, c.bill_length_mm, c.body_mass_g)
    .head(5)
)


species,island,bill_length_mm,body_mass_g
str,str,f64,f64
"""Adelie""","""Biscoe""",34.5,2900.0
"""Adelie""","""Biscoe""",36.5,2850.0
"""Adelie""","""Biscoe""",36.4,2850.0
"""Adelie""","""Dream""",33.1,2900.0


## 3. Set Membership (`in` and `not in`)

To match against a list of candidate values:
- Use SQL/Python-style `in` or `not in` in string expressions: `"species in ['Adelie', 'Chinstrap']"`
- Pass a candidate list or exclusion tuple in a dictionary: `{"species": ['Adelie', 'Chinstrap']}`, `{"sex": ("not in", ['Male'])}`

In [9]:
# Set membership via c.col.is_in(...)
(
    penguins
    .filter(c.species.is_in(["Adelie", "Chinstrap"]), c.island.is_in(["Dream", "Biscoe"]))
    .by("species", "island")
    .agg(n=c.species.count(), mean_mass=c.body_mass_g.mean())
)


species,island,n,mean_mass
str,str,u32,f64
"""Adelie""","""Biscoe""",44,3709.659091
"""Adelie""","""Dream""",56,3688.392857
"""Chinstrap""","""Dream""",68,3733.088235


In [10]:
# Membership and exclusion using ~ negation
(
    penguins
    .filter(c.species.is_in(["Adelie", "Gentoo"]), ~c.island.is_in(["Torgersen"]))
    .by("species", "island")
    .agg(n=c.species.count())
)


species,island,n
str,str,u32
"""Adelie""","""Biscoe""",44
"""Adelie""","""Dream""",56
"""Gentoo""","""Biscoe""",124


In [11]:
# Combining membership and exclusion
(
    penguins
    .filter(~c.species.is_in(["Gentoo"]), c.island.is_in(["Dream", "Torgersen"]))
    .select(c.species, c.island, c.sex, c.body_mass_g)
    .head(5)
)


species,island,sex,body_mass_g
str,str,str,f64
"""Adelie""","""Torgersen""","""Male""",3750.0
"""Adelie""","""Torgersen""","""Female""",3800.0
"""Adelie""","""Torgersen""","""Female""",3250.0
"""Adelie""","""Torgersen""",null,null
"""Adelie""","""Torgersen""","""Female""",3450.0


## 4. Boolean Logic: `and`, `or`, and Parentheses `(...)`

`pytae` supports rich boolean expressions in filter strings with:
- `and` for logical conjunction
- `or` for logical disjunction
- Arbitrary nesting with parentheses `(...)` for explicit precedence control

> [!NOTE]
> Following the Zen of Python (*"There should be one-- and preferably only one --obvious way to do it"*), `pytae` strictly uses `and` and `or`. Bitwise `&` and `|` are disallowed, and commas `,` are strictly punctuation for function arguments and lists, not boolean operators.

In [12]:
# Logical conjunction with multiple arguments or &
(
    penguins
    .filter(c.species == "Gentoo", c.body_mass_g > 5500, c.sex == "Female")
    .select(c.species, c.island, c.sex, c.body_mass_g)
    .head(5)
)


species,island,sex,body_mass_g
str,str,str,f64


In [13]:
# Logical disjunction with |
(
    penguins
    .filter((c.island == "Dream") | (c.island == "Torgersen"))
    .by("island")
    .agg(n=c.island.count())
)


island,n
str,u32
"""Dream""",124
"""Torgersen""",52


In [14]:
# Combining | and & with parentheses for precedence:
# Keep Adelie or Chinstrap penguins that weigh over 4000g:
(
    penguins
    .filter(((c.species == "Adelie") | (c.species == "Chinstrap")) & (c.body_mass_g > 4000))
    .select(c.species, c.island, c.sex, c.body_mass_g)
    .head(6)
)


species,island,sex,body_mass_g
str,str,str,f64
"""Adelie""","""Torgersen""","""Male""",4675.0
"""Adelie""","""Torgersen""",null,4250.0
"""Adelie""","""Torgersen""","""Male""",4400.0
"""Adelie""","""Torgersen""","""Male""",4500.0
"""Adelie""","""Torgersen""","""Male""",4200.0
"""Adelie""","""Dream""","""Male""",4150.0


In [15]:
# Complex nested logic:
# (Gentoo on Biscoe) OR (Adelie on Dream), with body mass >= 3500
(
    penguins
    .filter(
        (((c.species == "Gentoo") & (c.island == "Biscoe")) |
         ((c.species == "Adelie") & (c.island == "Dream"))) &
        (c.body_mass_g >= 3500)
    )
    .by("species", "island")
    .agg(n=c.species.count(), mean_mass=c.body_mass_g.mean())
)


species,island,n,mean_mass
str,str,u32,f64
"""Adelie""","""Dream""",33,3984.090909
"""Gentoo""","""Biscoe""",123,5076.01626


## 5. Range & Interval Filtering: Option 1 and Option 2

Previous versions of data tools frequently used mathematical interval strings like `'[3000, 4000]'` or `'(3000, 4000)'`. In modern `pytae`, string bracket notation was dropped to eliminate ambiguity with Python list literals and ensure unambiguous typing.

Instead, `pytae` provides two clean, powerful options for range filtering:

### Option 1: `("between", min, max)`
Pass `("between", min, max)` inside a dictionary to map directly to Polars' highly optimized native `.is_between(lower, upper)` method:
```python
df.filter({"body_mass_g": ("between", 3000, 4000)})
```

### Option 2: Multi-condition Operator Tuples `(('op1', val1), ('op2', val2))`
Pass a tuple of clean `(operator, value)` pairs inside a dictionary to apply multiple conditions on the same column without string-mashed syntax:
```python
# Inclusive bounds:
df.filter({"body_mass_g": ((">=", 3000), ("<=", 4000))})

# Exclusive or asymmetric bounds:
df.filter({"body_mass_g": ((">", 3000), ("<", 4500))})
df.filter({"bill_length_mm": ((">", 40), ("<", 45))})
```

### String Expressions with `and`
You can also write ranges naturally in filter strings:
```python
df.filter("body_mass_g >= 3000 and body_mass_g <= 4000")
```

In [16]:
# Range filtering via chained comparisons or bounds
display(
    penguins
    .filter((c.body_mass_g >= 3500) & (c.body_mass_g <= 3800))
    .select(c.species, c.island, c.body_mass_g)
    .head(5)
)


species,island,body_mass_g
str,str,f64
"""Adelie""","""Torgersen""",3750.0
"""Adelie""","""Torgersen""",3800.0
"""Adelie""","""Torgersen""",3650.0
"""Adelie""","""Torgersen""",3625.0
"""Adelie""","""Torgersen""",3700.0


In [17]:
# Clean separation of operators and values for asymmetric bounds:
(
    penguins
    .filter(
        c.bill_length_mm > 40,
        c.bill_length_mm < 45,
        c.body_mass_g >= 3500,
        c.body_mass_g <= 4000,
    )
    .select(c.species, c.bill_length_mm, c.body_mass_g)
    .head(5)
)


species,bill_length_mm,body_mass_g
str,f64,f64
"""Adelie""",40.6,3550.0
"""Adelie""",40.5,3950.0
"""Adelie""",40.9,3900.0
"""Adelie""",42.2,3550.0
"""Adelie""",40.8,3900.0


In [18]:
# Standard range using c expressions
(
    penguins
    .filter(c.body_mass_g >= 3500, c.body_mass_g <= 4000, c.bill_depth_mm > 18)
    .select(c.species, c.bill_depth_mm, c.body_mass_g)
    .head(5)
)


species,bill_depth_mm,body_mass_g
str,f64,f64
"""Adelie""",18.7,3750.0
"""Adelie""",20.6,3650.0
"""Adelie""",21.2,3800.0
"""Adelie""",18.7,3600.0
"""Adelie""",19.2,3800.0


## 6. Columns with Spaces & Bracket Notation `[col]`

When column names contain spaces or special punctuation (such as `bill length mm` or `counterparty_class_(irb)`):
- In **string expressions**, wrap the column name in square brackets: `[bill length mm] > 40`
- In **dictionaries**, use the column name directly as the key without any brackets: `{"bill length mm": "> 40"}`

In [19]:
# Create a DataFrame with spaces in column names
penguins_spaces = penguins.rename(columns={
    "bill_length_mm": "bill length mm",
    "body_mass_g": "body mass g",
})
print("Columns with spaces:", penguins_spaces.columns)


Columns with spaces: ['species', 'island', 'bill length mm', 'bill_depth_mm', 'flipper_length_mm', 'body mass g', 'sex']


In [20]:
# Filtering spaced columns using c("body mass g")
(
    penguins_spaces
    .filter(c("body mass g") > 5000, c.species == "Gentoo")
    .select(c.species, c.island, c("body mass g"))
    .head(5)
)


species,island,body mass g
str,str,f64
"""Gentoo""","""Biscoe""",5700.0
"""Gentoo""","""Biscoe""",5700.0
"""Gentoo""","""Biscoe""",5400.0
"""Gentoo""","""Biscoe""",5200.0
"""Gentoo""","""Biscoe""",5150.0


In [21]:
# Comparing spaced columns with c("...")
(
    penguins_spaces
    .filter((c("bill length mm") > 40) & (c("body mass g") < 4000))
    .select(c.species, c("bill length mm"), c("body mass g"))
    .head(5)
)


species,bill length mm,body mass g
str,f64,f64
"""Adelie""",40.3,3250.0
"""Adelie""",41.1,3200.0
"""Adelie""",40.6,3550.0
"""Adelie""",40.5,3200.0
"""Adelie""",40.5,3950.0


## 7. String Matching (`starts_with`, `ends_with`, `contains`, `regex`)

For text/string columns, `pytae` provides high-speed pattern matching inside dictionaries:
- `{"col": ('starts_with', prefix)}`: Matches string prefix
- `{"col": ('starts_with', [prefix1, prefix2, ...])}`: Matches any of several prefixes
- `{"col": ('ends_with', suffix)}` / `{"col": ('ends_with', [suffix1, suffix2, ...])}`: Matches suffixes
- `{"col": ('contains', substr)}`: Substring search
- `{"col": ('regex', pattern)}`: Regular expression pattern search

In [22]:
# Prefix and suffix matches via c.col.str
(
    penguins
    .filter(c.species.str.starts_with("Ad"), c.island.str.ends_with("oe"))
    .select(c.species, c.island, c.body_mass_g)
    .head(5)
)


species,island,body_mass_g
str,str,f64
"""Adelie""","""Biscoe""",3400.0
"""Adelie""","""Biscoe""",3600.0
"""Adelie""","""Biscoe""",3800.0
"""Adelie""","""Biscoe""",3950.0
"""Adelie""","""Biscoe""",3800.0


In [23]:
# Multiple prefixes: matches species starting with "Ad" OR "Ch"
(
    penguins
    .filter(c.species.str.starts_with("Ad") | c.species.str.starts_with("Ch"))
    .by("species")
    .agg(n=c.species.count())
)


species,n
str,u32
"""Adelie""",152
"""Chinstrap""",68


In [24]:
# Pattern matching (e.g. repeated "o")
(
    penguins
    .filter(c.species.str.contains("oo"))
    .distinct(c.species)
)


species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
str,str,f64,f64,f64,f64,str
"""Gentoo""","""Biscoe""",46.1,13.2,211.0,4500.0,"""Female"""


## 8. Missing and Null Checks (`is_null()`, `is_not_null()`)

In Pytae, all missing data and non-finite floating-point values (`NaN`) are normalized uniformly to Arrow `null` upon ingestion and derivation.

- **Canonical standard**: Use `.is_null()` and `.is_not_null()` across both string expressions and dictionaries.
- Dedicated `is_nan()` and `is_not_nan()` tokens are retired in favor of universal `is_null()` / `is_not_null()`, eliminating the silent IEEE 754 NaN comparison trap and ensuring consistent filtering.

In [25]:
# 1. Missing data checks via c.col.is_null() and is_not_null()
print("--- Missing sex rows (c.sex.is_null()) ---")
display(
    penguins
    .filter(c.sex.is_null())
    .select(c.species, c.island, c.body_mass_g, c.sex)
    .head(5)
)

print("--- Known sex rows (c.sex.is_not_null()) ---")
display(
    penguins
    .filter(c.sex.is_not_null())
    .select(c.species, c.island, c.body_mass_g, c.sex)
    .head(5)
)


--- Missing sex rows (c.sex.is_null()) ---


species,island,body_mass_g,sex
str,str,f64,str
"""Adelie""","""Torgersen""",null,null
"""Adelie""","""Torgersen""",3475.0,null
"""Adelie""","""Torgersen""",4250.0,null
"""Adelie""","""Torgersen""",3300.0,null
"""Adelie""","""Torgersen""",3700.0,null


--- Known sex rows (c.sex.is_not_null()) ---


species,island,body_mass_g,sex
str,str,f64,str
"""Adelie""","""Torgersen""",3750.0,"""Male"""
"""Adelie""","""Torgersen""",3800.0,"""Female"""
"""Adelie""","""Torgersen""",3250.0,"""Female"""
"""Adelie""","""Torgersen""",3450.0,"""Female"""
"""Adelie""","""Torgersen""",3650.0,"""Male"""


In [26]:
# 2. Spaced columns with null checks: c("...").is_not_null()
display(
    penguins_spaces
    .filter(c("bill length mm").is_not_null())
    .select(c.species, c.island, c("bill length mm"), c.sex)
    .head(5)
)


species,island,bill length mm,sex
str,str,f64,str
"""Adelie""","""Torgersen""",39.1,"""Male"""
"""Adelie""","""Torgersen""",39.5,"""Female"""
"""Adelie""","""Torgersen""",40.3,"""Female"""
"""Adelie""","""Torgersen""",36.7,"""Female"""
"""Adelie""","""Torgersen""",39.3,"""Male"""


In [27]:
# 3. Floating-point NaN checks via c.score.is_null():
df_measurements = pt.DataFrame({
    "sample_id": ["S1", "S2", "S3", "S4"],
    "score": [98.5, float("nan"), None, 76.2],
})

print("--- Original DataFrame (has both float NaN and missing null) ---")
display(df_measurements)

print("--- Null check: c.score.is_null() ---")
display(df_measurements.filter(c.score.is_null()))

print("--- Not null check: c.score.is_not_null() ---")
display(df_measurements.filter(c.score.is_not_null()))


--- Original DataFrame (has both float NaN and missing null) ---


sample_id,score
str,f64
"""S1""",98.5
"""S2""",null
"""S3""",null
"""S4""",76.2


--- Null check: c.score.is_null() ---


sample_id,score
str,f64
"""S2""",null
"""S3""",null


--- Not null check: c.score.is_not_null() ---


sample_id,score
str,f64
"""S1""",98.5
"""S4""",76.2


In [28]:
# 4. Combining null checks and predicates:
display(
    penguins
    .filter(c.sex.is_not_null(), c.body_mass_g > 4500)
    .select(c.species, c.body_mass_g, c.sex)
    .head(5)
)


species,body_mass_g,sex
str,f64,str
"""Adelie""",4675.0,"""Male"""
"""Adelie""",4650.0,"""Male"""
"""Adelie""",4600.0,"""Male"""
"""Adelie""",4700.0,"""Male"""
"""Adelie""",4725.0,"""Male"""


## 9. Column-to-Column Comparisons

You can compare two DataFrame columns directly against each other (e.g., detecting limit breaches or verifying ID reconciliations) in both Python and CLI using bare column names without any `col()` wrapper:

In [29]:
# Comparing two columns directly using canonical c expressions
df_risk = pt.DataFrame({
    "exposure": [120, 250, 80, 410],
    "limit": [100, 200, 150, 400],
    "tranche_id": ["T1", "T2", "T3", "T4"],
    "deal_id": ["T1", "D2", "T3", "D4"],
})

print("--- Breaches (c.exposure > c.limit) ---")
display(df_risk.filter(c.exposure > c.limit))

print("--- Reconciled (c.tranche_id == c.deal_id) ---")
display(df_risk.filter(c.tranche_id == c.deal_id))


--- Breaches (c.exposure > c.limit) ---


exposure,limit,tranche_id,deal_id
i64,i64,str,str
120,100,"""T1""","""T1"""
250,200,"""T2""","""D2"""
410,400,"""T4""","""D4"""


--- Reconciled (c.tranche_id == c.deal_id) ---


exposure,limit,tranche_id,deal_id
i64,i64,str,str
120,100,"""T1""","""T1"""
80,150,"""T3""","""T3"""


## 10. External Variable Injection (`@var` and `col(@var)`)

You can inject external local variables into filter expressions:
- `@var`: Injects a scalar threshold or comparison literal
- `col(@var)`: Injects a dynamic column name from a Python variable

In [30]:
# Inject python scalar variables and dynamic column references
threshold = 100
target_col = "deal_id"

display(df_risk.filter(c.exposure > threshold, c.tranche_id == c(target_col)))


exposure,limit,tranche_id,deal_id
i64,i64,str,str
120,100,"""T1""","""T1"""


## 11. Strict Error Handling & Diagnostic Feedback

`pytae` is intentionally designed with helpful, unambiguous error messages that guide you to the right syntax rather than silently failing or doing unexpected things.

In [31]:
# 1. Keyword arguments are strictly rejected (forces explicit c expressions)
try:
    penguins.filter(species="Gentoo")
except TypeError as err:
    print("Caught expected TypeError:\n", err)


Caught expected TypeError:
 filter(): keyword arguments are not supported (received ['species']). Equality comparisons strictly require '==' to avoid '=' assignment ambiguity. Use a string expression (e.g. df.filter("species == 'Gentoo'")) or a dictionary (e.g. df.filter({'species': 'Gentoo'}))


In [32]:
# 2. Legacy string expressions are strictly retired and raise TypeError
try:
    penguins.filter("species == 'Gentoo'")
except TypeError as err:
    print("Caught expected TypeError:\n", err)


Caught expected TypeError:
 String expressions in filter() are retired. Use canonical c expressions, e.g. df.filter(c.species == 'Adelie', c.body_mass_g > 3500)


In [33]:
# 3. String assignments are disallowed
try:
    penguins.filter("species = 'Gentoo'")
except TypeError as err:
    print("Caught expected TypeError:\n", err)


Caught expected TypeError:
 String expressions in filter() are retired. Use canonical c expressions, e.g. df.filter(c.species == 'Adelie', c.body_mass_g > 3500)


In [34]:
# 4. Raw string conditions are rejected
try:
    penguins.filter("species == 'Gentoo', body_mass_g > 5000")
except TypeError as err:
    print("Caught expected TypeError:\n", err)


Caught expected TypeError:
 String expressions in filter() are retired. Use canonical c expressions, e.g. df.filter(c.species == 'Adelie', c.body_mass_g > 3500)


In [35]:
# 5. Legacy interval strings are retired
try:
    penguins.filter("body_mass_g = [3000, 4000]")
except TypeError as err:
    print("Caught expected TypeError:\n", err)


Caught expected TypeError:
 String expressions in filter() are retired. Use canonical c expressions, e.g. df.filter(c.species == 'Adelie', c.body_mass_g > 3500)


In [36]:
# 6. Type safety in c comparisons
df_str = pt.DataFrame({"id": ["1", "9", "10"], "val": [100, 200, 300]})
display(df_str.filter(c.id == "9"))


id,val
str,i64
"""9""",200


In [37]:
# 7. Unregistered attribute reference suggestion or error
try:
    penguins.filter(c.speceis == "Adelie")
except (KeyError, Exception) as err:
    print("Caught expected error:\n", err)


Caught expected error:
 unable to find column "speceis"; valid columns: ["species", "island", "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "sex"]

Did you mean "species"?


## 12. Real-World Analytical Pipelines

Filter works seamlessly as the entry point or intermediate step in pytae pipelines combining filtering, grouping, aggregation, reshaping, and sorting.

In [38]:
# Tips dataset: weekend dinner parties of 2+ spending > 
(
    tips
    .filter(c.day.is_in(["Sat", "Sun"]), c.time == "Dinner", c.size >= 2, c.total_bill > 15)
    .by("smoker")
    .agg(mean_tip=c.tip.mean(), mean_bill=c.total_bill.mean(), total_diners=c.size.sum(), n_tables=c.tip.count())
    .arrange(c.mean_tip.desc())
)


smoker,mean_tip,mean_bill,total_diners,n_tables
cat,f64,f64,i64,u32
"""No""",3.521081,23.257297,225,74
"""Yes""",3.384889,26.112,122,45


In [39]:
# Titanic dataset: survival rate of passengers in 1st/2nd class by sex
(
    titanic
    .filter(c.pclass.is_in([1, 2]), c.age.is_not_null(), (c.age >= 18) & (c.age <= 60))
    .by("pclass", "sex")
    .agg(survival_rate=c.survived.mean(), n_passengers=c.survived.count())
    .arrange(c.pclass.asc(), c.sex.asc())
)


pclass,sex,survival_rate,n_passengers
i64,str,f64,u32
1,"""female""",0.973333,75
1,"""male""",0.411765,85
2,"""female""",0.903226,62
2,"""male""",0.058824,85
